# Readmission Risk — Model Training

Trains an XGBoost classifier to predict 30-day hospital readmission, and evaluates it
with metrics appropriate for an imbalanced clinical outcome.

**Prerequisite:** run `notebooks/01_diabetes_preprocessing.ipynb` first so
`data/processed_diabetes.csv` exists.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve, auc,
                               classification_report, confusion_matrix, ConfusionMatrixDisplay)
import xgboost as xgb
import joblib
import os

os.makedirs("models", exist_ok=True)

df = pd.read_csv("data/processed_diabetes.csv")
print(f"Loaded: {df.shape}")

## 1. Encode categorical features

XGBoost needs numeric input. We ordinal-encode categoricals (simple, works well with
tree models — unlike linear models, trees don't assume any ordering meaning between
encoded values, they just learn splits).

In [ ]:
y = df["readmit_30d"]
X = df.drop(columns=["readmit_30d"])

cat_cols = X.select_dtypes(include="object").columns.tolist()
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = encoder.fit_transform(X[cat_cols])

print(f"Encoded {len(cat_cols)} categorical columns")
print(f"Feature matrix: {X.shape}")

## 2. Train/test split

We stratify on the target to keep the same ~8.8% positive rate in both splits.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.4f}")
print(f"Test positive rate:  {y_test.mean():.4f}")

## 3. Train XGBoost

30-day readmission is rare (~9% of patients) — a model that just predicts "no
readmission" for everyone would already be 91% "accurate" while being clinically
useless. We use `scale_pos_weight` to make the model pay proportionally more attention
to the minority (positive) class during training.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print("Training complete.")

## 4. Evaluate

For imbalanced clinical outcomes, **AUROC and AUPRC matter far more than accuracy**.
AUPRC (area under the precision-recall curve) is especially informative here since it's
sensitive to how the model performs specifically on the rare positive class.

In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

auroc = roc_auc_score(y_test, y_pred_proba)
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
auprc = auc(recall, precision)

print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["No readmit", "Readmit <30d"]))

## 5. ROC and Precision-Recall curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, label=f"AUROC = {auroc:.3f}", color="#2563eb", linewidth=2)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve — Readmission Model")
axes[0].legend()

axes[1].plot(recall, precision, label=f"AUPRC = {auprc:.3f}", color="#dc2626", linewidth=2)
axes[1].axhline(y=y_test.mean(), linestyle="--", color="gray", label="Baseline (prevalence)")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve — Readmission Model")
axes[1].legend()

plt.tight_layout()
plt.savefig("models/diabetes_roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No readmit", "Readmit <30d"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix — Readmission Model (threshold = 0.5)")
plt.tight_layout()
plt.savefig("models/diabetes_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Feature importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
importances.sort_values().plot(kind="barh", ax=ax, color="#2563eb")
ax.set_title("Top 15 Feature Importances — Readmission Model")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("models/diabetes_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save the model

In [ ]:
joblib.dump(model, "models/diabetes_readmission_model.pkl")
joblib.dump(encoder, "models/diabetes_encoder.pkl")
joblib.dump(list(X.columns), "models/diabetes_feature_names.pkl")
print("Saved model, encoder, and feature names to models/")